# Evaluation

Builds three retrieval methods (keyword, vector, hybrid) and evaluates them against a synthetic ground-truth set (hit rate / MRR), then compares two RAG prompt variants via LLM-as-judge.

This is the trimmed, reproducible version of the evaluation work described in `README.md` (summary + results tables) and `PLAN.md` (full narrative and decision history, including dead ends not shown here).

Re-running this top to bottom reproduces the search-evaluation numbers exactly (it loads the committed `data/ground_truth.csv` rather than regenerating it). The v1/v2 LLM-eval numbers are historical - see the note near `INSTRUCTIONS_V1` below for why.

# Initialize

In [34]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
from dotenv import load_dotenv
from openai import OpenAI   
import pandas as pd
load_dotenv()
openai_client = OpenAI()

from minsearch import Index

True

In [2]:
with open("data/anime.jsonl") as f:
    documents = [json.loads(line) for line in f]

# Search

In [8]:
from minsearch import Index

for doc in documents:
    doc["genres_text"] = " ".join(doc["genres"])
    doc["tags_text"] = " ".join(doc["tags"])

index = Index(text_fields=["title_romaji", "title_english", "description", "genres_text", "tags_text"])
index.fit(documents)

# Vector index

In [1]:
import json
import numpy as np
from tqdm.auto import tqdm
from minsearch import VectorSearch
from embedder import Embedder

In [3]:
embed = Embedder()

texts = [doc["description"] for doc in documents]

In [4]:
batch_size = 50
X = []
for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    X.extend(embed.encode_batch(batch))

X = np.array(X)

  0%|          | 0/87 [00:00<?, ?it/s]

In [5]:
vindex = VectorSearch()
vindex.fit(X, documents)

# Hybrid search

In [10]:
def rrf(search_results, k=1, num_results=10):
    scores = {}
    doc_map = {}

    for results in search_results:
        for rank, doc in enumerate(results):
            key = doc["id"]
            if key not in scores:
                scores[key] = 0
                doc_map[key] = doc
            scores[key] += 1 / (k + rank + 1)

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_map[key] for key, _ in ranked[:num_results]]


def hybrid_search(query, num_results=10):
    keyword_results = index.search(query, num_results=num_results)
    vector_results = vindex.search(embed.encode(query), num_results=num_results)
    return rrf([keyword_results, vector_results], num_results=num_results)

# Ground truth

In [17]:
from pydantic import BaseModel
from openai import OpenAI
from dotenv import load_dotenv
from evaluation_utils import llm_structured, calc_total_price

In [18]:
load_dotenv()
openai_client = OpenAI()

class Questions(BaseModel):
    questions: list[str]

In [58]:
data_gen_instructions = """
You emulate someone trying to find an anime whose title they don't remember.
Based on the anime record below, write 3 short, natural-sounding descriptions
of the plot, setting, or vibe that this person might type into a search box.
Make the 3 descriptions distinct from each other - vary which detail they
focus on (plot event, setting, tone/vibe, a character situation, etc).

Do not mention the anime's title, character names, studio, or any other
unique proper nouns. Describe it the way someone would if they only
remembered the general premise - not exact wording from the synopsis.
Dont incorporate multiple features into a single description.

Do not create long questions mixing multiple ideas, emulate a user
remembering only a couple of key features of the anime.
Keep it casual, like a real search query, not overly formal, and avoid
complete sentences.
""".strip()

In [51]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)
    out, usage = llm_structured(openai_client, data_gen_instructions, user_prompt, Questions)

    records = [{"question": q, "document": doc["id"]} for q in out.questions]
    return records, usage


In [63]:
import random
random.seed(0)
sample = random.sample(documents, 5)

for doc in sample:
    records, usage = generate_ground_truth(doc)
    print(doc["title_romaji"], "(pop:", doc["popularity"], ")")
    for r in records:
        print("  -", r["question"])

Mugen no Juunin (pop: 15374 )
  - immortal samurai who has to kill a bunch of evil guys to die
  - dark historical sword fighting anime with a cursed swordsman
  - samurai revenge story traveling with a girl after her family gets killed
Tenchi Muyou! Ryououki Dai 1-ki (pop: 12468 )
  - guy accidentally frees a space pirate girl from a cave
  - normal boy ends up living with a bunch of alien girls
  - old-school sci-fi comedy with a guy surrounded by girls fighting over him
FAIRY TAIL (2014) (pop: 185180 )
  - magic guild tournament anime with dragons and a hooded stranger
  - rowdy fantasy adventurers in a medieval guild, lots of fights and comedy
  - anime about a found-family magic team facing a big final trial
DIABOLIK LOVERS MORE,BLOOD (pop: 31329 )
  - vampire brothers at school, dark romance anime
  - girl living with a bunch of scary vampire guys
  - twisted supernatural love triangle with vampires and bullying
Gekidol (pop: 7553 )
  - post-apocalyptic idol girls using holograms

## Generate for sample (100)

In [65]:
import random

random.seed(1)
sample = random.sample(documents, 100)

### Generate in parallel

In [74]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, sample, generate_ground_truth)

ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

100%|██████████| 100/100 [00:26<00:00,  3.80it/s]


300

In [75]:
calc_total_price(usages)

0.06875175000000003

In [76]:
df_ground_truth = pd.DataFrame(ground_truth)
df_ground_truth.to_csv("data/ground_truth.csv", index=False)

**Reproducibility note**: re-running the cells above calls the LLM again and will produce *different* synthetic queries (generation isn't deterministic), which would change the exact numbers below. The evaluation cells that follow load `data/ground_truth.csv` (committed to the repo) rather than the in-memory `ground_truth` from this section, so the reported results reproduce exactly without needing to regenerate ground truth.

# Search evaluation

In [78]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

def text_search(query):
    return index.search(query, num_results=5)

def vector_search(query):
    return vindex.search(embed.encode(query), num_results=5)

def hybrid_search_fn(query):
    return hybrid_search(query, num_results=5)

In [79]:
from tqdm import tqdm

def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])
    return [int(d["id"] == doc_id) for d in results]

def compute_relevance_total(ground_truth, search_function):
    return [compute_relevance(q, search_function) for q in tqdm(ground_truth)]

def hit_rate(relevance):
    return sum(1 for line in relevance if 1 in line) / len(relevance)

def mrr(relevance):
    total_score = 0.0
    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score += 1 / (rank + 1)
                break
    return total_score / len(relevance)

def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)
    return {"hit_rate": hit_rate(relevance_total), "mrr": mrr(relevance_total)}

In [80]:
print("keyword:", evaluate(ground_truth, text_search))
print("vector: ", evaluate(ground_truth, vector_search))
print("hybrid: ", evaluate(ground_truth, hybrid_search_fn))

100%|██████████| 300/300 [00:01<00:00, 191.11it/s]


keyword: {'hit_rate': 0.21666666666666667, 'mrr': 0.1201111111111111}


100%|██████████| 300/300 [00:02<00:00, 138.33it/s]


vector:  {'hit_rate': 0.3233333333333333, 'mrr': 0.23022222222222222}


100%|██████████| 300/300 [00:03<00:00, 95.68it/s]

hybrid:  {'hit_rate': 0.3333333333333333, 'mrr': 0.19783333333333342}


In [81]:
k_values = [1, 5, 10, 20, 60]

cache = []
for q in tqdm(ground_truth):
    kw = text_search(q["question"])
    vec = vector_search(q["question"])
    cache.append((q, kw, vec))

for k in k_values:
    relevance_total = []
    for q, kw, vec in cache:
        doc_id = q["document"]
        fused = rrf([kw, vec], k=k, num_results=5)
        relevance_total.append([int(d["id"] == doc_id) for d in fused])

    print(f"k={k}:", {"hit_rate": hit_rate(relevance_total), "mrr": mrr(relevance_total)})

100%|██████████| 300/300 [00:03<00:00, 82.65it/s]


k=1: {'hit_rate': 0.3333333333333333, 'mrr': 0.19783333333333342}
k=5: {'hit_rate': 0.3333333333333333, 'mrr': 0.20116666666666672}
k=10: {'hit_rate': 0.3333333333333333, 'mrr': 0.20116666666666672}
k=20: {'hit_rate': 0.3333333333333333, 'mrr': 0.20116666666666672}
k=60: {'hit_rate': 0.3333333333333333, 'mrr': 0.20116666666666672}


# Query rewriting

In [31]:
from query_rewrite import rewrite_query

for q in ground_truth[:5]:
    rewritten = rewrite_query(openai_client, q["question"])
    print("RAW:      ", q["question"])
    print("REWRITTEN:", rewritten)
    print()

RAW:       magic school anime where the hero goes back in time to fix the future
REWRITTEN: A magic school anime centered on a hero who travels back in time in an effort to fix a disastrous future. The story likely blends fantasy, school-life, and time-travel elements, with the protagonist navigating an academy setting full of magical training, rivalries, and hidden dangers while trying to change events before they lead to ruin. The tone suggests an action-driven, dramatic premise with a focus on fate, second chances, and the consequences of altering the timeline.

RAW:       guy wakes up years in the past after the world ends, goes back to mage academy
REWRITTEN: A man awakens years in the past after experiencing the end of the world, finding himself given a rare second chance to relive his life. With the apocalypse looming in his memory, he returns to a mage academy, where he must navigate magical training, rebuilding relationships, and the growing sense of impending disaster. The st

In [33]:
def vector_search_with_rewrite(query):
    rewritten = rewrite_query(openai_client, query)
    return vindex.search(embed.encode(rewritten), num_results=5)

print("vector + rewrite:", evaluate(ground_truth, vector_search_with_rewrite))
print("vector (baseline):", evaluate(ground_truth, vector_search))

100%|██████████| 300/300 [08:21<00:00,  1.67s/it]


vector + rewrite: {'hit_rate': 0.19666666666666666, 'mrr': 0.14555555555555555}


100%|██████████| 300/300 [00:02<00:00, 125.38it/s]


vector (baseline): {'hit_rate': 0.29333333333333333, 'mrr': 0.2228333333333334}


**Verdict: evaluated, not shipped.** Query rewriting made retrieval substantially worse, not better - hit rate dropped from 0.293 to 0.197, MRR from 0.223 to 0.146 (both computed in the same run against the same ground-truth snapshot, so the comparison itself is apples-to-apples even though the baseline number here differs from the `vector: 0.323` figure earlier in this notebook - ground truth was regenerated between those two runs, see the reproducibility note above).

**Why this likely happens**: the ground-truth queries are deliberately short and sparse ("emulate a user remembering only a couple of key features", no proper nouns - see the generation instructions above). The rewrite step expands these into long, generic synopsis-style paragraphs (see the spot-check above - "magic school anime where the hero goes back in time" becomes a multi-sentence paragraph about "fate, second chances, and the consequences of altering the timeline"). That expansion dilutes the embedding: a short, specific phrase concentrates its vector on the few distinctive details that actually matter for matching; a longer, generically-worded paragraph averages across a lot of stock phrasing that doesn't correspond to how AniList actually writes synopses. More words turned out to mean more noise, not more signal.

`app.py` continues shipping plain `VectorIndexAdapter` (unchanged) rather than `RewritingVectorIndexAdapter` (`search_backends.py`) - the rewriting code exists and is genuinely evaluated above, it's just not the better retrieval method for this corpus.

# Rag assembly

In [85]:
from openai import OpenAI
from dotenv import load_dotenv
from rag_helper import RAGBase
from search_backends import VectorIndexAdapter

In [86]:
vector_index = VectorIndexAdapter(vindex, embed)

# LLM-as-a-judge

In [90]:
INSTRUCTIONS_V2 = '''
Your task is to help a user find anime based on a description of the
plot, vibe, or themes they're looking for.

Look at the candidate anime provided (retrieved by searching synopses,
genres, and tags). Pick exactly ONE best-matching title from the
candidates - do not hedge or list multiple options as equally likely.

End your answer with a final line in this exact format:
ANSWER: <title>

Briefly justify your pick using only the retrieved information, then
give the ANSWER line. If none of the candidates are a good match, still
pick the closest one but say so in your justification.
'''.strip()

`INSTRUCTIONS_V1` is hardcoded here rather than relying on `RAGBase`'s default: the default in `rag_helper.py` was later changed in place to the precise/v2 prompt once it was picked as the winner, so the original open-ended prompt this comparison was run against no longer exists anywhere else in the codebase.

In [ ]:
INSTRUCTIONS_V1 = '''
Your task is to help a user find anime based on a description of the
plot, vibe, or themes they're looking for.

Use the provided candidate anime (retrieved by searching synopses,
genres, and tags) to answer. Recommend the best-matching title(s) from
the candidates and briefly explain why they match, grounded only in
the retrieved information. If none of the candidates are a good match,
say so honestly instead of making one up.
'''.strip()

In [91]:
from judge import evaluate_answer

id_to_title = {doc["id"]: doc["title_romaji"] for doc in documents}

rag_v1 = RAGBase(index=vector_index, llm_client=openai_client, instructions=INSTRUCTIONS_V1)
rag_v2 = RAGBase(index=vector_index, llm_client=openai_client, instructions=INSTRUCTIONS_V2)

In [92]:
random.seed(2)
eval_sample = random.sample(ground_truth, 30)

In [96]:
def run_variant(rag, sample):
    records = []
    usages = [] 
    for q in tqdm(sample):
        answer = rag.rag(q["question"])
        correct_title = id_to_title[q["document"]]
        eval_result, usage = evaluate_answer(openai_client, q["question"], correct_title, answer)
        records.append({"score": eval_result.score, "reasoning": eval_result.reasoning})
        usages.append(usage)
    return records, usages

In [102]:
results_v1, usages_v1 = run_variant(rag_v1, eval_sample)
results_v2, usages_v2 = run_variant(rag_v2, eval_sample)
print(calc_total_price(usages_v1))
print(calc_total_price(usages_v2))

100%|██████████| 30/30 [01:34<00:00,  3.14s/it]

0.021419999999999995
0.01868025


In [103]:
for label, results in [("v1 (current)", results_v1), ("v2 (forced single answer)", results_v2)]:
    good = sum(1 for r in results if r["score"] == "good")
    print(f"{label}: {good}/{len(results)} = {good/len(results):.1%}")

v1 (current): 14/30 = 46.7%
v2 (forced single answer): 13/30 = 43.3%
